# Laser-Plasma Absorption Coefficients
Sanitized public snapshot of an exploratory notebook.

When a laser interacts with a plasma, we can expect at least three laser absorption mechanisms: electron-neutral inverse bremsstrahlung, electron-ion inverse bremsstrahlung, and single photon photoionization.

## Photoionization
The absorption coefficient for photoionization in a single species plasma is given by:

### <center>$\alpha_{PI} = \sum_{z=0}^{Z_{max}} \sum_{j=N_*^z}^{N^z_{max}} n^z_{j} \sigma^z_{PI,j}(T)$ 

where $z$ is the charge state of the particle, $Z_{max}$ is the maximum charge state considered, $N_*^Z$ and $N^Z_{max}$ are minimum and maximum energy levels considered for charge state $z$, $n^Z_{j}$ is the number density of particles in energy level $j$ with charge state $z$ and $\sigma^Z_{PI,j}(T)$  is the cross section for photoionization of energy level $j$ with charge state $z$.

The photoionization cross section for a charge state $z$, $\sigma^z_{PI,j}(T)$, is calculated with:

###  <center>$\sigma^z_{PI,j}(T) = \frac{32 \pi^2 (z+1)^2 e^6 k_e^3}{3 \sqrt{3} h^4 c \nu^3} \frac{u^{z+1}(T)}{g^z_j} \frac{dE^z_j}{dj}$
    
This equations uses SI units where e is electron charge, $k_e$ is the Coulomb constant, $h$ is Planck's constant, $\nu$ is the laser frequency, c is the speed of light, $u^{z+1}(T)$ is the parition function of the $z+1$ charge state, $g^z_j$ is the statistical weight of energy level $j$ in charge state $z$, and $\frac{dE^z_j}{dj}$ is the spacing between energy levels.

In order to calculate this cross section and the relevant partition function $u^{z+1}(T)$, we need to obtain the atomic energy levels from the NIST Atomic Spectra Database (https://physics.nist.gov/PhysRefData/ASD/levels_form.html).

I have already done this for Cu and saved the data as text files for Cu I, Cu II, and Cu III (corresponding to $Cu^{0}$, $Cu^{1+}$, and $Cu^{2+}$, respectively.) These data files contain the energy of each level, relative the the ground state of that charge state, the statistical weights $g_j^z$, and other information.

Below, I define a function to clean up the NIST data files, and load each file into a DataFrame.

In [ ]:
# Standard imports for loading data, calculations, and plotting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

def process_NIST_file(file):
    """
    This function takes in a tab delimited NIST atomic spectra database file,
    removes problematic characters in the energy level column, and returns a 
    pandas DataFrame.
    """
    df = pd.read_csv(file, delimiter='\t')
    E = np.zeros(len(df))
    for i in range(len(df)):
        s = df['Level (eV)'][i]
        if type(s) != str:
            E[i] = s
        elif '[' in s:
            E[i] = float(s.replace('[','').replace(']',''))
        elif '?' in s:
            E[i] = float(s.replace('?',''))
        else:
            E[i] = float(s)
    df['Level (eV)'] = E
    return df#df.drop(0)

# Save each NIST data file as a DataFrame
Cu_I_levels = process_NIST_file('SeI Energy Levels.txt')
Cu_II_levels = process_NIST_file('SeII Energy Levels.txt')
Cu_III_levels = process_NIST_file('SeIII Energy Levels.txt')
Cu_IV_levels = process_NIST_file('SeIV Energy Levels.txt')

The first few rows of the data files look like this:

In [ ]:
Cu_I_levels.head()

Now that we have this atomic data, we can calculate the partition function and fractional occupation of each energy level. The parition function is calculated with the equation:
### <center>$u^z(T) = \sum_{j=0}^{N^z_{max}} g_j^z e^{-\big(\frac{E_j^z-E_0^z}{k_B T}\big)}$
where $k_B$ is the Boltzmann constant, T is the temperature, $E_j^z$ is the energy of level $j$ at charge state $z$, $E_0^z$ is the ground level of the charge state, which we will take to be 0 since our data files have the energy levels relative to the charge state ground level.
    Lastly the fractional occupation of energy level $j$ can be calculated with the partition function and is given by:
    
### <center>$ x_j^z = \frac{1}{u^z(T)} g_j^z e^{\frac{-E_j^z}{k_B T}}$
    
Below, I define functions to calculate these quantities and visualize the results.

In [ ]:
def partition_function(df, T):
    """
    This function takes in a DataFrame created from a NIST data file df,
    a temperature T, and returns the parition function. This function 
    produces identical results to the NIST databse when the same number 
    of energy levels are used (I checked).
    """
    
    k_eV = 8.6173324e-5 # eV K-1
    E_j = df['Level (eV)'] # eV
    
    u = np.sum(df['g']*np.exp(-E_j/(k_eV*T)))

    return u

def fractional_occupation(df, T):
    """
    The fractional occupation is calculated using a DataFrame created from
    a NIST data file and returns an array of values.
    """
    k_eV = 8.6173324e-5 # eV K-1
    u = partition_function(df,T)
    g_j = df['g']
    E_j = df['Level (eV)'] # eV
    x_j = (1/u)*g_j*np.exp(-E_j/(k_eV*T))
    
    return np.array(x_j)   

We can visualize the fractional occupations for Cu I at various temperatures and confirm that the sum of the fractional occupations is equal to one.

__Try changing the temperatures, or charge state to view the results for other scenarios.__

In [ ]:
Temperatures = np.array([0.8,1.1,1.8])*1.160451812e4 # temperatures in eV, converted to K

charge_state = Cu_I_levels # Choose either Cu_I_levels, Cu_II_levels, or Cu_III_levels

for i in Temperatures:
    frac_occ = fractional_occupation(charge_state, i)
    plt.semilogy(frac_occ, label='T={:.1f} eV'.format(i/1.160451812e4))
    summ = (np.sum(frac_occ))
    print('The sum for T = {:.1f} eV is {:.3f}'.format(i/1.160451812e4, summ))
plt.xlabel('Energy Level j')
plt.ylabel('Fractional Occupation')
plt.legend()
plt.show()

Next, we can use the partition function and NIST data to calculate the photoionization cross section for each energy level in a single charge state, $\sigma^z_{PI,j}(T)$. I define the function below.

In [ ]:
def sigma_PI(Z, T, df1, df2):
    """
    Returns the photoionization cross section for a charge state Z at temperature T. 
    Input the DataFrames for charge state Z as df1 and Z+1 as df2
    """
    e = 1.602176634e-19   # C
    c = 299792458.0       # m s-1
    lamb = 248e-9         # m
    nu = c/lamb           # s-1
    h = 6.6260700408e-34  # kg m2 s-1
    ke = 8.9875517923e9   # kg m3 s-2 C-2
    k_eV = 8.6173324e-5   # ev K-1
    
    g_z1 = df1['g'].values
    E_z1 = df1['Level (eV)'].values*e # converts eV to J

    dEdj = np.zeros(len(E_z1))
    for i in range(1, len(E_z1)):
        dEdj[i] = E_z1[i] - E_z1[i-1]

    u_z2 = partition_function(df2, T)
    
    PI_sig =((32*np.pi**2*(Z+1)**2*e**6*ke**3)/(3*np.sqrt(3)*h**4*c*nu**3))*(u_z2/g_z1)*(dEdj) # m2
    return PI_sig

In [ ]:
Ts = np.linspace(300,100000,100)
sigmas0 = np.zeros(len(Ts))
sigmas1 = np.zeros(len(Ts))
sigmas2 = np.zeros(len(Ts))

for i in range(len(Ts)):
    sigmas0[i] = np.sum(sigma_PI(0, Ts[i], Cu_I_levels, Cu_II_levels)[5:])
    sigmas1[i] = np.sum(sigma_PI(1, Ts[i], Cu_II_levels, Cu_III_levels)[53:])
    sigmas2[i] = np.sum(sigma_PI(2, Ts[i], Cu_III_levels, Cu_IV_levels)[52:])

In [ ]:
plt.semilogy(Ts, sigmas0, 'r')
plt.semilogy(Ts, sigmas1, 'g')
plt.semilogy(Ts, sigmas2, 'b')

plt.xlabel('Temperature (K)')
plt.ylabel('Cross Section (m$^2$)')
plt.xlim(0,100000)
#plt.ylim(1e-21, 1e-18) #upper should be 1e-18
plt.grid(which='both')

Now, the cross section and fractional occupation can be multiplied to show the the photoionization cross section for each energy level j, weighted by the fractional occupation at temperature T. Below, I plot this quantity for various temperatures for the Cu$^0$ to Cu$^{1+}$ and Cu$^{1+}$ to Cu$^{2+}$ transitions.

In [ ]:
Temperatures = np.array([0.5,1,1.5])*1.160451812e4 # temperatures in eV, converted to K

charge_state = Cu_I_levels # Choose either Cu_I_levels, Cu_II_levels, or Cu_II_levels

fig, ax = plt.subplots(1,2, figsize=(10,5))

for i in Temperatures:
    ax[0].semilogy(sigma_PI(0,i,Cu_I_levels, Cu_II_levels)*fractional_occupation(Cu_I_levels, i),
              label='T={:.1f} eV'.format(i/1.160451812e4))
    ax[0].set_xlabel('Energy Level j')
    ax[0].set_ylabel('$\sigma_{PI,j}^0 x_j^0$ (m$^2$)', fontsize=15)
    ax[0].set_xlim(0,75)

    
    ax[1].semilogy(sigma_PI(1,i+4,Cu_II_levels, Cu_III_levels)*fractional_occupation(Cu_II_levels, i+4),
              label='T={:.1f} eV'.format((i/1.160451812e4)+4))
    ax[1].set_xlabel('Energy Level j')
    ax[1].set_ylabel('$\sigma_{PI,j}^1 x_j^1$ (m$^2$)', fontsize=15)
    ax[1].set_xlim(0,75)

ax[0].legend()
ax[1].legend()

fig.tight_layout()

The last step is to mulitply the above quantity by the total density of the charge state and sum all energy states j to get the absorption coefficient for photoionization. By multiplying by the total density of the charge state with the fractional occupation, we get the number density for each energy level $n_j^z$.

So the sake of demonstration, we will pretend we have a density of Cu$^0$ n$_0$ = 10$^{18} m^{-3}$ at a temperature T = 0.5 eV. First, I will plot $\alpha_{PI,j}$ which is the absorption coefficient per energy level. After, I will sum $\alpha_{PI,j}$ to get the total absorption coefficient for these neutral species, $\alpha_{PI}$

When we sum the individual coefficients, we need to make sure we exclude states that our 5 eV (248 nm) photons can't ionize. For Cu, this means that the minimum level $N_*^z$ has to have energy greater than or equal to the ionization energy of state $z$ minus the photon energy. I will define this below as well.

$E(N_*^z)_{min} >= IP^z-h\nu$

In [ ]:
h_eV = 4.135667696e-15 # Planck constant eV s
c = 299792458.0        # m s-1

photon_energy = h_eV*c/248e-9 # eV

Cu_IP1, Cu_IP2 = 7.7264, 20.29240

minimum_energy_Cu_I = Cu_IP1-photon_energy  #eV
minimum_energy_Cu_II = Cu_IP2-photon_energy #eV

min_energy_level_Cu_I = Cu_I_levels['Level (eV)'][Cu_I_levels['Level (eV)']>=minimum_energy_Cu_I].index[0]
min_energy_level_Cu_II = Cu_II_levels['Level (eV)'][Cu_II_levels['Level (eV)']>=minimum_energy_Cu_II].index[0]


print('Minimum energy in level for Cu I is {} eV at level j={}'.format(minimum_energy_Cu_I,min_energy_level_Cu_I))
print('Minimum energy in level for Cu II is {} eV at level j={}'.format(minimum_energy_Cu_II,min_energy_level_Cu_II))

n0 = 1e18 # m-3
T = 0.5*1.160451812e4 

# absorption coefficient per energy level
alpha_PI_n0_j = sigma_PI(0,T,Cu_I_levels, Cu_II_levels)*fractional_occupation(Cu_I_levels, T)*n0

plt.semilogy(alpha_PI_n0_j)
plt.vlines(min_energy_level_Cu_I, min(alpha_PI_n0_j), max(alpha_PI_n0_j), color='k', linestyle='--')
plt.xlabel('Energy Level j')
plt.ylabel('$\\alpha_{PI,j}$ (m$^{-1}$)', fontsize=15)
plt.ylim(min(alpha_PI_n0_j), max(alpha_PI_n0_j))
plt.show()

alpha_PI_n0 = np.sum(alpha_PI_n0_j[Cu_I_levels['Level (eV)']>=minimum_energy_Cu_I])

print('The total absorption coefficient for these neutral species, alpha_IP = {:.3e} m-1'.format(alpha_PI_n0))

The dashed line shows the lowest energy level that a 5 eV photon could ionize.

Lastly, I will define a function that will calculate the total $\alpha_{IP}$ considering the $z=0$ and $z=1$ charge states.

In [ ]:
def alpha_PI(n0, ni1, T):
    """
    This function calculates the photoionization absorption coefficent for 5eV photons
    when given the density of neutral and singly ionized species, and temperature. 
    First, the fractional occupations of the atomic energy levels for Cu^0 and Cu^{1+}
    are calculated with the given temperature. Next, the cross sections for each charge
    state are calculated. Finally, the 
    """
    
    x_z0_j = fractional_occupation(Cu_I_levels, T) # fractional occupation for Cu^0
    x_z1_j = fractional_occupation(Cu_II_levels, T) # fractional occupation for Cu^{1+}
    #x_z2_j = fractional_occupation(Cu_III_levels, T) # fractional occupation for Cu^{2+}
    
    j_min_I = 5 #for Se#5# for Te#3 #for Cu# minimum level for Cu^0
    j_min_II = 53 # for Se#54 for Te#83 # for Cu# minimum level for Cu^{1+}
    #j_min_III = 328 # minimum level for Cu^{2+}
    
    sigma_PI_z0 = sigma_PI(0, T, Cu_I_levels, Cu_II_levels) # cross section per level for Cu^0
    sigma_PI_z1 = sigma_PI(1, T, Cu_II_levels, Cu_III_levels) # cross section per level for Cu^1+
    #sigma_PI_z2 = sigma_PI(2, T, Cu_III_levels, Cu_IV_levels) # cross section per level for Cu^2+
    
    # To use this in our existing laser code, maybe instead of 
    alpha_z0_j =sigma_PI_z0*x_z0_j*n0 # absorption coefficient per level for Cu^0
    alpha_z1_j = sigma_PI_z1*x_z1_j*ni1 # absorption coefficient per level for Cu^{1+}
    #alpha_z2_j = sigma_PI_z2*x_z2_j*ni2 # absorption coefficient per level for Cu^{2+}
    
    alpha_z0 = np.sum(alpha_z0_j[j_min_I:]) # total absorption from Cu^0, considering the minimum energy level
    alpha_z1 = np.sum(alpha_z1_j[j_min_II:]) # total absorption from Cu^{1+}, considering the minimum energy level
    #alpha_z2 = np.sum(alpha_z2_j[j_min_III:]) # total absorption from Cu^{2+}, considering the minimum energy level
    
    alpha_PI_total =  alpha_z0 + alpha_z1# + alpha_z2 # sum of individual coefficients from each charge state

    return alpha_PI_total

In [ ]:
print("{:.6e}".format(alpha_PI(1e26,1e21,20000)))

## Electron-Neutral Inverse Bremsstrahlung

Electron-neutral inverse bremsstrahlung absorption is given by the equation:

### <center> $\alpha_{IB,en} = \bigg( 1 - e^{-\frac{hc}{\lambda k_B T}} \bigg) Q(T) n_e n_0$
    
where all units are cgs. $Q(T)$ is the electron-neutral collision cross section and is calculated as:

### <center> $Q(T) = \int_0^{\infty} K_a(E) f_{MB}(E,T) dE$

where $K_a(E)$ is:

### <center> $K_a(E) = \frac{2e^2 k_e}{3 \pi m_e c \nu^2} \sqrt{\frac{2(E+h\nu)}{m_e}} \frac{E+h\nu}{h\nu} \sigma_{T}(E+h\nu)$

and $f_{MB}$ is:

### <center> $f_{MB}(E, T) = 2 \sqrt{\frac{E}{\pi (k_b T)^3}} e^{\frac{-E}{k_B T}}$

In the equation for $K_a(E)$, $\sigma_{T}$ is the momentum transport cross section for electron neutral collisions.

In the cell below, I define a function to calculate the differential $Q(E,T)$, i.e. not integrated with respect to energy yet. For now, I am using a $\sigma_T$ that is constant with energy because this data is not available without requesting it from other researchers. I am using $\sigma_T = 2.5x10^{-19}$ which is a kind of average value for Cu that I have estimated from this paper https://journals.aps.org/pra/abstract/10.1103/PhysRevA.82.062703.

After using this function to calculate the differential Q, it needs to be integrated. In python, I am using the scipy.integrate.quad function and integrating E = 1 to 10$^{12}$ eV to a relative error or $1x10^{-12}$ which I have found gives accurate results in a reasonable time.

In [ ]:
from scipy.integrate import quad

def get_energy(MTcrossfile):
    MTcrossdata = pd.read_csv(MTcrossfile, delim_whitespace=True)
    return MTcrossdata['E_eV']

def differential_Q(E,T):
    """
    This function takes in an electron energy array (for integration) and a temperature
    and returns Q(E,T). Q(E,T) needs to be integrated w.r.t E in order to get the Q
    needed for the electron-neutral IB coefficient.
    """
    e = 1.606176634e-19
    me = 9.1093837015e-31
    c = 299792458.0
    e0 = 8.8541878128e-12
    lamb = 248e-9
    nu = c/lamb
    h = 6.6260700408e-34
    k = 1.380648813e-23 #8.617e-5
    ke = 8.9875517923e9
    
    def sig(E):
        """
        This subroutine should contain a function that gives back the 
        correct value of $\sigma_T$ for a given energy.
        
        Currently, it just returns a fixed value for all E.
        """

        a02 = 5.29177210903e-11**2
    
        MTcrossdata = pd.read_csv("MT_01_01", delim_whitespace=True)
        sig = MTcrossdata['MT(a_0^2)']*a02
        E0 = MTcrossdata['E_eV']*e

        idx = np.argmin(np.abs(E0 - E))
        #return 2.5e-19
        return sig[idx]
    #K_IB = np.zeros(len(E0))
    #max_boltz = np.zeros(len(E0))
        
    #for i in range(len(E)):
    max_boltz = 2*np.sqrt(E/(np.pi*(k*T)**3)) * np.exp(-E/(k*T))
        #K_IB = ((e**2)/(12*np.pi**2*me*c*nu**2*e0))*np.sqrt((2*(E[i]+h*nu))/(me)) \
        #    * ((E[i]+h*nu)/(h*nu)*sig(E[i],MTcrossfile) + (E)/(h*nu)*sig(E[i],MTcrossfile))
    K_IB = (2*e**2*ke)/(3*np.pi*me*c*nu**2) * np.sqrt((2*(E+h*nu))/(me)) \
                    * (E+h*nu)/(h*nu) * sig(E+h*nu)
    return K_IB*max_boltz #, K_IB, max_boltz

In [ ]:
file = 'MT_01_01'
E = get_energy(file)*1.602e-19

plt.plot(E,differential_Q(E, 11600))

In [ ]:
# Integration example with scipy.integrate.quad

Ts = np.arange(300,10*11605,1000)
vals, errs = np.zeros(len(Ts)), np.zeros(len(Ts))

for i in range(len(Ts)):
    vals[i], errs[i] = quad(differential_Q, 0,1e12/6.242e18,
                    epsabs=0, epsrel=1e-12,
                    args=(Ts[i]), points=20/6.242e18)
    print(i, 'of', len(Ts))
        
#print("Q = {:.3e} m^5".format(val))
#print("Estimated error = {:.3e}".format(err))

In [ ]:
plt.errorbar(Ts,vals)

In [ ]:
MTcrossdata = pd.read_csv('MT_01_01', delim_whitespace=True)
a02 = 5.29177210903e-11**2
sig = MTcrossdata['MT(a_0^2)']*a02
E0 = MTcrossdata['E_eV']

plt.semilogy(E0, sig)
print(E0[589],sig[589])

Lastly, I will define a function to calculate $\alpha_{IB,en}$, using the integration routine to calculate Q(T) for every point.

In [ ]:
def alpha_IBen(T, n0, ne):
    # densities are input in m-3
    #cgs units
    me = 9.1094e-28    # g
    e = 4.8032e-10      # cm3/2 g1/2 s-1
    h = 6.62606957e-27  # cm2 g s-1
    c = 2.9979e10       # cm/s
    k = 1.3806488e-16   # cm2 g s-2 K-1
    
    lamb = 248e-9*100
    

    Q_in_meters5, err = quad(differential_Q, 0,1e2/6.242e18,epsabs=0, epsrel=1e-12,args=(T))
    Q = Q_in_meters5*1e10
    
    # uncomment below to use constant value for Q
    #Q = 5e-40 #cm5
    
    alpha_IBen = (1. - np.exp(-h*c/(lamb*k*T)))*Q*(ne/1e6)*(n0/1e6)*100. #m-1
    return alpha_IBen

## Electron-Ion Inverse Bremsstrahlung

The last absorption mechanism we include is the electron-ion inverse bremsstrahlung term. This is dependent on the density of ions and electrons and is given by the equation:

### <center> $\alpha_{IB,ei} = \big(1-e^{-\frac{h c}{\lambda k_B T}} \big) \frac{4 e^6 \lambda^3}{3 h c^4 m_e}\
                                \sqrt{\frac{2 \pi}{3 m_e k_B T}} \
                                 n_e \sum_{z=1}^{Z_{max}} z^2 n^z$
    
where $n^z$ is the density of charge state z, and $n_e$ is the electron density. As with the electron-neutral term, these units are cgs but converted to SI later. We only consider $z=1$ and $z=2$ in our code.

In [ ]:
def alpha_IBei(T, ni1, ni2, ne):
    #cgs units
    me = 9.1094e-28    # g
    e = 4.8032e-10      # cm3/2 g1/2 s-1
    h = 6.62606957e-27  # cm2 g s-1
    c = 2.9979e10       # cm/s
    k = 1.3806488e-16   # cm2 g s-2 K-1

    lamb = 248e-9*100
    
    c = (1. - np.exp(-h*c/(lamb*k*T)))*4*e**6*lamb**3/(3.*h*c**4*me)* \
        np.sqrt(2.*np.pi/(3.*me*k*T))*(ne/1e6)*(1**2*(ni1/1e6) + 2**2*(ni2/1e6))* \
        100. #m-1   
    return c

## Comparison of Results with and without Photoionization

One of the major critisisms of our adopted laser ablation model is that the the electron-neutral IB coefficient is incorrectly calculated. Specifically, we used $Q = 10^{-36} cm^5$. This value is inflated by about 4 orders of magnitude over what modern measurements of this cross section report. To fix this, we can calculate $Q$ in the way I described above. This brings the magnitude of $Q$ down to the order of 10$^{-40} cm^5$ which is not strong enough of an interaction to trigger plasma shielding at the low laser powers we use (I checked with some simulations).

It turns out that it was proved in some experiments that photoionization is actually the predominent mechanism for plasma shielding and so it is important to consider it in our simulations. 

Below, I load in data from the peak of the laser pulse for Te (one of the figures in our current draft of the chalcogen paper) and recalculate the absorption coefficients to include photoionization for comparison.

In [ ]:
# load in data
Te_17_5 = pd.read_csv(r"Te_vs_x_25ns.csv")

x = Te_17_5['Points:0']*1e6
n0 = Te_17_5['n0']
ni1 = Te_17_5['ni1']
ni2 = Te_17_5['ni2']
ne = Te_17_5['ne']
T = Te_17_5['T']

In [ ]:
alpha_PI_x = np.zeros(len(T))
alpha_IBei_x = np.zeros(len(T))
alpha_IBen_x = np.zeros(len(T))

for i in range(len(T)):
    alpha_PI_x[i] = alpha_PI(n0[i], ni1[i], T[i])
    alpha_IBei_x[i] = alpha_IBei(T[i], ni1[i], ni2[i], ne[i])            
    alpha_IBen_x[i] = alpha_IBen(T[i], n0[i], ne[i])
    print(i, 'of ', len(T))

plt.semilogy(x, alpha_PI_x, label='PI', linewidth=3)
plt.plot(x, alpha_IBen_x, label='my e-n IB',linewidth=3)
plt.plot(x, Te_17_5['alphaIBen'], label='model e-n IB',linewidth=3)
plt.plot(x, alpha_IBei_x, label='my e-i IB')
plt.plot(x, Te_17_5['alphaIBei'],'o', label='model e-i IB')
plt.ylim(1e-14, 1e7)
plt.xlim(0,55)
plt.ylabel('$\\alpha$ (m$^{-1}$)')
plt.xlabel('Distance ($\mu$m)')
plt.legend()
plt.show()

From the plot above, all the absorption coefficient plotted together. PI is the photoionization terms, "my e-n IB" and "my e-i IB" are calculated using the functions in this notebook while "model e-n IB" and "model e-i IB" were calculated during the simulation.

Interestingly, the model's $\alpha_{IB,en}$ using the inflated value of $Q=10^{-36} cm^5$ and $\alpha_{PI}$ behave similarly, with similar profiles in space (at this time) and a strong peak near 10 $\mu m$.

#### Sanity Check
The electron-ion term is identical between my calculations and the model, so that gives some confidence there. I'll also check that my electron-neutral term is the same as the model when using a constant $Q=10^{-36} cm^5$. And they are identical, see in the cell below.

In [ ]:
def alpha_IBen_check(T, n0, ne):
    # densities are input in m-3
    #cgs units
    me = 9.1094e-28    # g
    e = 4.8032e-10      # cm3/2 g1/2 s-1
    h = 6.62606957e-27  # cm2 g s-1
    c = 2.9979e10       # cm/s
    k = 1.3806488e-16   # cm2 g s-2 K-1
    
    lamb = 248e-9*100
    

    #Q_in_meters5, err = quad(differential_Q, 0,1e2/6.242e18,epsabs=0, epsrel=1e-12,args=(T))
    #Q = Q_in_meters5*1e10
    
    # uncomment below to use constant value for Q
    Q = 1e-36 #cm5
    
    alpha_IBen = (1. - np.exp(-h*c/(lamb*k*T)))*Q*(ne/1e6)*(n0/1e6)*100. #m-1
    return alpha_IBen

alpha_IBen_x_test = np.zeros(len(T))
for i in range(len(T)):          
    alpha_IBen_x_test[i] = alpha_IBen_check(T[i], n0[i], ne[i])

plt.semilogy(x, alpha_IBen_x_test, label='my e-n IB Q=10$^{-36}$',linewidth=3)
plt.plot(x, Te_17_5['alphaIBen'], '--', label='model e-n IB',linewidth=3)
plt.ylim(1e-14, 1e7)
plt.xlim(0,55)
plt.ylabel('$\\alpha$ (m$^{-1}$)')
plt.xlabel('Distance ($\mu$m)')
plt.legend()
plt.show()

In [ ]:
Ts = np.linspace(0,100000,1000)
sigmas0 = np.zeros(len(Ts))
sigmas1 = np.zeros(len(Ts))
sigmas2 = np.zeros(len(Ts))

for i in range(len(Ts)):
    sigmas0[i] = np.trapz(sigma_PI(0, Ts[i], Cu_I_levels, Cu_II_levels)[3:])
    sigmas1[i] = np.trapz(sigma_PI(1, Ts[i], Cu_II_levels, Cu_III_levels)[83:])
    sigmas2[i] = np.trapz(sigma_PI(1, Ts[i], Cu_III_levels, Cu_IV_levels)[328:])
    if i%250==0:
        print('step %d of 1000' %(i))

In [ ]:
plt.semilogy(Ts, sigmas0)
plt.semilogy(Ts, sigmas1)
plt.semilogy(Ts, sigmas2)

plt.xlabel('Temperature (K)')
plt.ylabel('Cross Section (m$^2$)')
plt.ylim(1e-21, 1e-15) #upper should be 1e-18

In [ ]:
np.sum(sigma_PI(0, 4000, Cu_I_levels, Cu_II_levels)[3:])